# Taller Pandas: 120 años de historia olímpica

**Curso:** Programación II, Ciencia de Datos
**Grupo:** taller-pandas-olimpiadas-grupoX

En este notebook analizamos los datos de Kaggle *120 years of Olympic history: athletes and results* (`athlete_events.csv` y `noc_regions.csv`). Seguimos el orden de los 14 puntos del taller: carga, descripción, nuevas variables, fusión, datos faltantes, preguntas con `groupby` y tablas de contingencia, gráficos y conclusiones.

> Cada fila de `athlete_events.csv` es **un atleta en una prueba de una edición de los Juegos**, no un atleta único. Lo tenemos en cuenta en todo el análisis.

Los archivos originales deben estar en la carpeta `data/` (ver `data/README.md`). Las tablas se guardan en `outputs/tablas/` y los gráficos en `outputs/graficos/`.

In [1]:
# Librerías y rutas del proyecto
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

DATA = "../data"
TABLAS = "../outputs/tablas"
GRAFICOS = "../outputs/graficos"
os.makedirs(TABLAS, exist_ok=True)
os.makedirs(GRAFICOS, exist_ok=True)

## 1. Carga de los datos

Cargamos cada archivo en un dataframe distinto: `atletas` y `regiones`.

In [2]:
# Punto 1: carga de cada dataset en un dataframe distinto
atletas = pd.read_csv(f"{DATA}/athlete_events.csv")
regiones = pd.read_csv(f"{DATA}/noc_regions.csv")

print("atletas:", atletas.shape)
print("regiones:", regiones.shape)
atletas.head()

atletas: (271116, 15)
regiones: (230, 3)


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [3]:
regiones.head()

,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN


`atletas` tiene **271.116 filas y 15 columnas**, y `regiones` tiene **230 filas y 3 columnas**.

## 2. Descripción de las variables

**athlete_events.csv** (`atletas`)

| Variable | Tipo | Descripción |
|---|---|---|
| ID | entero | Identificador único de cada atleta |
| Name | texto | Nombre del atleta |
| Sex | categórica | Sexo: M (hombre) o F (mujer) |
| Age | numérica | Edad en años al momento de la prueba |
| Height | numérica | Estatura en centímetros |
| Weight | numérica | Peso en kilogramos |
| Team | texto | Nombre del equipo con el que compitió |
| NOC | categórica | Código de tres letras del Comité Olímpico Nacional |
| Games | categórica | Edición de los Juegos (año + temporada) |
| Year | entero | Año de la edición |
| Season | categórica | Temporada: Summer (verano) o Winter (invierno) |
| City | texto | Ciudad sede |
| Sport | categórica | Deporte |
| Event | categórica | Prueba específica dentro del deporte |
| Medal | categórica | Gold, Silver, Bronze o vacío (NA) si no ganó |

**noc_regions.csv** (`regiones`)

| Variable | Tipo | Descripción |
|---|---|---|
| NOC | categórica | Código del Comité Olímpico Nacional (llave para unir) |
| region | texto | País o región a la que corresponde el NOC |
| notes | texto | Notas aclaratorias (por ejemplo, nombres históricos como Bohemia o Yugoslavia) |

In [4]:
# Punto 2: tipos de dato y valores no nulos de cada variable
atletas.info()
print()
regiones.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
dtypes: float64(3), int64(2), str(10)
memory usage: 31.0 MB

<class 'pandas.DataFrame'>
RangeIndex: 230 entries, 0 to 229
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   NOC     230 non-null    s

Las variables numéricas son `ID`, `Age`, `Height`, `Weight` y `Year` (aunque `ID` y `Year` funcionan más como identificadores que como medidas). Todo lo demás es texto o categoría. Desde ya se ve que `Age`, `Height`, `Weight` y `Medal` tienen menos valores no nulos que el total de filas, y en `regiones` faltan algunos valores de `region` (3) y casi todas las `notes`.

## 3. Estadísticas básicas y comparación entre dataframes

In [5]:
# Punto 3: estadísticas básicas de las variables numéricas de atletas
atletas.describe().round(2)

,ID,Age,Height,Weight,Year
count,271116.00,261642.00,210945.00,208241.00,271116.00
mean,68248.95,25.56,175.34,70.70,1978.38
std,39022.29,6.39,10.52,14.35,29.88
min,1.00,10.00,127.00,25.00,1896.00
25%,34643.00,21.00,168.00,60.00,1960.00
50%,68205.00,24.00,175.00,70.00,1988.00
75%,102097.25,28.00,183.00,79.00,2002.00
max,135571.00,97.00,226.00,214.00,2016.00


In [6]:
# Estadísticas de las variables categóricas de atletas
atletas.describe(exclude="number")

,Name,Sex,Team,NOC,Games,Season,City,Sport,Event,Medal
count,271116,271116,271116,271116,271116,271116,271116,271116,271116,39783
unique,134732,2,1184,230,51,2,42,66,765,3
top,Robert Tait McKenzie,M,United States,USA,2000 Summer,Summer,London,Athletics,Football Men's Football,Gold
freq,58,196594,17847,18853,13821,222552,22426,38624,5733,13372


In [7]:
# Estadísticas de regiones
regiones.describe(include="all")

,NOC,region,notes
count,230,227,21
unique,230,206,21
top,AFG,Germany,Netherlands Antilles
freq,1,4,1


In [8]:
# Comparación entre dataframes a través de la llave NOC
noc_atletas = set(atletas["NOC"].unique())
noc_regiones = set(regiones["NOC"].unique())

print("NOC distintos en atletas:", len(noc_atletas))
print("NOC distintos en regiones:", len(noc_regiones))
print("NOC en atletas que no están en regiones:", sorted(noc_atletas - noc_regiones))
print("NOC en regiones que no están en atletas:", sorted(noc_regiones - noc_atletas))
print("Regiones distintas:", regiones["region"].nunique())
print("Equipos (Team) distintos en atletas:", atletas["Team"].nunique())

NOC distintos en atletas: 230
NOC distintos en regiones: 230
NOC en atletas que no están en regiones: ['SGP']
NOC en regiones que no están en atletas: ['SIN']
Regiones distintas: 206
Equipos (Team) distintos en atletas: 1184


**Interpretación**

- La edad promedio por registro es de **25,6 años** (mediana 24), con un rango muy amplio: de 10 a 97 años.
- La estatura promedio es de **175,3 cm** y el peso promedio de **70,7 kg**. El 50 % central de los atletas mide entre 168 y 183 cm.
- Los años van de **1896 a 2016**, pero la mediana es 1988, lo que significa que la mitad de los registros son de los últimos ~30 años.
- Hay **134.732 nombres distintos**, 1.184 equipos, 230 NOC, 51 ediciones, 66 deportes y 765 pruebas. El 72,5 % de los registros son hombres (196.594 de 271.116).
- **Comparación:** ambos dataframes tienen 230 NOC, pero no son exactamente los mismos: `SGP` (Singapur) está en atletas y no en regiones, y `SIN` está en regiones y no en atletas. Además, los 230 NOC se agrupan en solo **206 regiones**, porque varios códigos históricos apuntan al mismo país (por ejemplo, GER, FRG, GDR y SAA corresponden a Germany). `atletas` tiene mucho más detalle (1.184 valores de `Team`) que `regiones`, que solo traduce códigos.

## 4. Variable IMC

$$IMC = \frac{Weight}{(Height/100)^2}$$

In [9]:
# Punto 4: variable IMC = peso / (estatura en metros)^2
atletas["IMC"] = atletas["Weight"] / (atletas["Height"] / 100) ** 2
atletas["IMC"].describe().round(2)

count    206853.00
mean         22.78
std           2.91
min           8.36
25%          20.96
50%          22.53
75%          24.21
max          63.90
Name: IMC, dtype: float64

In [10]:
# IMC promedio por deporte (deportes con al menos 500 registros con IMC)
imc_deporte = (atletas.groupby("Sport")["IMC"]
               .agg(["count", "mean", "std", "min", "max"])
               .query("count >= 500")
               .sort_values("mean")
               .round(2))
imc_deporte.to_csv(f"{TABLAS}/imc_por_deporte.csv")
pd.concat([imc_deporte.head(5), imc_deporte.tail(5)])

,count,mean,std,min,max
Sport,,,,,
Rhythmic Gymnastics,615,17.29,1.31,13.33,22.04
Synchronized Swimming,849,19.67,1.42,13.49,37.29
Triathlon,526,20.39,1.32,17.21,25.28
Figure Skating,1512,20.84,2.15,15.06,28.88
Ski Jumping,1968,20.90,1.92,17.19,28.44
Ice Hockey,4579,25.14,1.94,17.69,33.52
Judo,3384,25.59,4.69,15.15,63.90
Baseball,846,25.68,2.29,17.09,33.77
Bobsleigh,2206,26.99,2.53,17.96,39.54


**¿Qué opinamos del IMC?**

El IMC promedio es de **22,8** y la mitad de los registros está entre 21,0 y 24,2, es decir, en el rango que normalmente se considera "peso normal". Aun así, el IMC no tiene el mismo sentido en todos los deportes:

- Es una medida muy simple, porque no distingue entre masa muscular y grasa. Un levantador de pesas (IMC promedio de **27,7**) o un judoca (**25,6**) tiene un IMC alto por músculo, no por sobrepeso.
- En deportes estéticos o de resistencia es bajo: gimnasia rítmica **17,3**, natación sincronizada **19,7** y triatlón **20,4**.
- En deportes con categorías de peso (judo, halterofilia, lucha) la desviación es alta (4,7 y 5,5) porque se mezclan pesos muy distintos. Por eso el máximo de **63,9** aparece en judo.
- En deportes como tiro, vela o las competencias de arte, el físico casi no influye en el resultado, así que el IMC dice poco.
- Además, solo **206.853** registros tienen IMC, porque falta la estatura o el peso en cerca del 24 % de los casos.

Conclusión: el IMC sirve para describir el tipo de cuerpo de cada deporte, pero no como indicador de salud y no es comparable entre todos los deportes.

## 5. Fusión (merge) con regiones

Usamos `how="left"` porque queremos **conservar todas las filas de atletas** aunque su NOC no aparezca en `regiones`. Un `inner` habría eliminado esas filas y un `outer` habría agregado filas de regiones sin atletas (como `SIN`).

In [11]:
# Punto 5: fusión con how='left' para conservar todas las filas de atletas
filas_antes = len(atletas)
df = atletas.merge(regiones, on="NOC", how="left")
filas_despues = len(df)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("¿Se conservan las filas?", filas_antes == filas_despues)
print("Filas sin región:", df["region"].isna().sum())

Filas antes: 271116
Filas después: 271116
¿Se conservan las filas? True
Filas sin región: 370


In [12]:
# NOC de las filas que quedaron sin región
df[df["region"].isna()].groupby(["NOC", "notes"], dropna=False).size().reset_index(name="filas")

,NOC,notes,filas
0,ROT,Refugee Olympic Team,12
1,SGP,NaN,349
2,TUV,Tuvalu,7
3,UNK,Unknown,2


El número de filas **no cambia** (271.116 antes y después), así que el merge no duplicó filas; esto confirma que `NOC` es única en `regiones`. Quedaron **370 filas sin región**. ¿Por qué ocurre?

- **SGP (349 filas):** Singapur está codificado como `SGP` en atletas y como `SIN` en regiones. Es un error de consistencia entre las dos fuentes, así que lo corregimos abajo.
- **ROT (12), TUV (7) y UNK (2):** el código sí existe en `regiones`, pero su `region` está vacía a propósito, porque son el *Refugee Olympic Team*, Tuvalu y un equipo *Unknown*. No corresponden a un país del listado.

In [13]:
# Corrección: Singapur aparece como SGP en atletas y como SIN en regiones
df.loc[df["NOC"] == "SGP", "region"] = "Singapore"
print("Filas sin región después de corregir SGP:", df["region"].isna().sum())

Filas sin región después de corregir SGP: 21


Después de corregir Singapur, solo quedan **21 filas sin región** (ROT, TUV y UNK).

## 6. Variable binaria `medalla`

Los vacíos de `Medal` **no son datos perdidos**: significan que el atleta no ganó medalla en esa prueba. Por eso los reemplazamos por `"Sin medalla"` y creamos `medalla` = 1 si ganó oro, plata o bronce, y 0 en otro caso.

In [14]:
# Punto 6: variable binaria medalla (los vacíos de Medal significan que no ganó)
df["Medal"] = df["Medal"].fillna("Sin medalla")
df["medalla"] = (df["Medal"] != "Sin medalla").astype(int)
df["Medal"].value_counts()

Medal
Sin medalla    231333
Gold            13372
Bronze          13295
Silver          13116
Name: count, dtype: int64

In [15]:
# Filas frente a atletas únicos
print("Filas (atleta-prueba-edición):", len(df))
print("Atletas únicos (ID):", df["ID"].nunique())
print("Promedio de filas por atleta:", round(len(df) / df["ID"].nunique(), 2))

Filas (atleta-prueba-edición): 271116


Atletas únicos (ID): 135571
Promedio de filas por atleta: 2.0


In [16]:
# Efecto de los deportes de equipo en el conteo de medallas
medallas_filas = df["medalla"].sum()
medallas_reales = (df[df["medalla"] == 1]
                   .drop_duplicates(["Games", "Event", "Medal"])
                   .shape[0])
print("Medallas contando filas:", medallas_filas)
print("Medallas contando una por (edición, prueba, tipo):", medallas_reales)

ejemplo = df[(df["Event"] == "Basketball Men's Basketball") & (df["Games"] == "2016 Summer") & (df["Medal"] == "Gold")]
print("Filas de oro en baloncesto masculino 2016:", len(ejemplo), "-> pero es una sola medalla para", list(ejemplo["Team"].unique()))

Medallas contando filas: 39783
Medallas contando una por (edición, prueba, tipo): 18293
Filas de oro en baloncesto masculino 2016: 12 -> pero es una sola medalla para ['United States']


Hay **39.783 registros con medalla** (13.372 oros, 13.116 platas y 13.295 bronces) y 231.333 sin medalla.

Pero el dataframe tiene **271.116 filas** y solo **135.571 atletas únicos**, es decir, en promedio cada atleta aparece 2 veces (porque compite en varias pruebas o en varias ediciones). Esto tiene dos implicaciones al contar medallas:

1. Si contamos filas, contamos **medallas por atleta**, no medallas entregadas. En los deportes de equipo cada integrante tiene su propia fila: el oro de baloncesto masculino 2016 aparece **12 veces** aunque Estados Unidos recibió una sola medalla.
2. Si contamos una medalla por combinación (edición, prueba, tipo de medalla), el total baja de 39.783 a **18.293**, que se acerca más a la cantidad de medallas entregadas como en un medallero oficial (no es exacto, porque hay pruebas con dos bronces o empates).

Por eso, en el resto del taller hablamos de **"registros con medalla"** y comparamos **tasas** (medallas / registros), que no se distorsionan tanto por el tamaño de los equipos.

## 7. Estadísticas de nuevo y representatividad de los datos

In [17]:
# Punto 7: estadísticas nuevamente con las variables nuevas
df[["Age", "Height", "Weight", "Year", "IMC", "medalla"]].describe().round(2)

,Age,Height,Weight,Year,IMC,medalla
count,261642.00,210945.00,208241.00,271116.00,206853.00,271116.00
mean,25.56,175.34,70.70,1978.38,22.78,0.15
std,6.39,10.52,14.35,29.88,2.91,0.35
min,10.00,127.00,25.00,1896.00,8.36,0.00
25%,21.00,168.00,60.00,1960.00,20.96,0.00
50%,24.00,175.00,70.00,1988.00,22.53,0.00
75%,28.00,183.00,79.00,2002.00,24.21,0.00
max,97.00,226.00,214.00,2016.00,63.90,1.00


In [18]:
# Participación por época (década), temporada y sexo
df["decada"] = (df["Year"] // 10) * 10
part_decada = pd.crosstab(df["decada"], df["Sex"])
part_decada["Total"] = part_decada.sum(axis=1)
part_decada["% mujeres"] = (100 * part_decada["F"] / part_decada["Total"]).round(1)
part_decada.to_csv(f"{TABLAS}/participacion_por_decada_sexo.csv")
part_decada

Sex,F,M,Total,% mujeres
decada,,,,
1890,0,380,380,0.0
1900,107,7964,8071,1.3
1910,87,3953,4040,2.2
1920,832,14727,15559,5.3
1930,918,9804,10722,8.6
1940,761,6719,7480,10.2
1950,2821,12971,15792,17.9
1960,5675,23519,29194,19.4
1970,5214,17247,22461,23.2


In [19]:
# Participación por temporada y sexo
part_temp = pd.crosstab(df["Season"], df["Sex"], margins=True, margins_name="Total")
print(part_temp)
print()
print((100 * pd.crosstab(df["Season"], df["Sex"], normalize="index")).round(1))
print()
print("Ediciones por temporada:")
print(df.groupby("Season")["Games"].nunique())

Sex         F       M   Total
Season                       
Summer  59443  163109  222552
Winter  15079   33485   48564
Total   74522  196594  271116

Sex        F     M
Season            
Summer  26.7  73.3
Winter  31.0  69.0

Ediciones por temporada:
Season
Summer    29
Winter    22
Name: Games, dtype: int64


**¿Son los datos representativos de toda la historia?** No de forma pareja:

- **Por época:** la participación crece muchísimo con el tiempo. La década de 1890 tiene 380 registros y la de 2000 tiene 49.357. La mediana del año es 1988, así que la información está cargada hacia los Juegos modernos. Las décadas de 1910 y 1940 tienen pocos registros por las guerras mundiales (no hubo Juegos en 1916, 1940 ni 1944).
- **Por sexo:** el porcentaje de mujeres pasa de **0 %** en 1896 a **1,3 %** en 1900, **17,9 %** en los 50 y **44,3 %** en la década de 2010. Cualquier conclusión sobre mujeres está dominada por los Juegos recientes.
- **Por temporada:** el verano tiene **222.552** registros (82 %) frente a **48.564** del invierno, y además 29 ediciones contra 22. Los Juegos de Invierno empezaron en 1924 y son más pequeños. La proporción de mujeres es un poco mayor en invierno (31,0 % vs 26,7 %).
- La media de `medalla` es 0,15: aproximadamente el **15 %** de los registros terminó con medalla.

En resumen, los datos sí cubren 120 años, pero la mayor parte de la información (y la más completa) es de la segunda mitad del siglo XX en adelante, de los Juegos de Verano y de hombres.

## 8. Datos faltantes

In [20]:
# Punto 8: datos faltantes por variable
conteo = df.isna().sum()
conteo["Medal"] = atletas["Medal"].isna().sum()
faltantes = pd.DataFrame({"faltantes": conteo, "porcentaje": (100 * conteo / len(df)).round(2)})
faltantes = faltantes[faltantes["faltantes"] > 0].sort_values("faltantes", ascending=False)
faltantes.to_csv(f"{TABLAS}/datos_faltantes.csv")
faltantes

,faltantes,porcentaje
notes,266077,98.14
Medal,231333,85.33
IMC,64263,23.70
Weight,62875,23.19
Height,60171,22.19
Age,9474,3.49
region,21,0.01


In [21]:
# Proporción de faltantes antes y después de 1960
df["epoca"] = np.where(df["Year"] < 1960, "Antes de 1960", "1960 en adelante")
falt_epoca = (100 * df.groupby("epoca")[["Age", "Height", "Weight", "IMC", "region"]]
              .agg(lambda s: s.isna().mean())).round(2)
falt_epoca.to_csv(f"{TABLAS}/faltantes_por_epoca.csv")
falt_epoca

,Age,Height,Weight,IMC,region
epoca,,,,,
1960 en adelante,0.53,5.32,5.66,5.92,0.01
Antes de 1960,13.47,79.06,82.28,83.62,0.00


In [22]:
# Porcentaje de faltantes de estatura y peso por década
(100 * df.groupby("decada")[["Age", "Height", "Weight"]].agg(lambda s: s.isna().mean())).round(1)

,Age,Height,Weight
decada,,,
1890,42.9,87.9,87.1
1900,30.4,86.9,88.7
1910,3.9,82.2,85.2
1920,19.0,82.5,87.9
1930,5.1,77.6,85.7
1940,15.7,84.4,84.6
1950,5.8,69.1,69.2
1960,1.4,6.1,6.6
1970,0.7,5.2,5.8


**¿Qué variables tienen faltantes y qué significan?**

| Variable | % faltante | Significado |
|---|---|---|
| Medal | 85,3 % | No es un faltante real: significa que **no ganó medalla** |
| notes | 98,1 % | Solo se escribe cuando el NOC necesita una aclaración |
| Weight / Height | 23,2 % / 22,2 % | No se registró la medida, sobre todo en los primeros Juegos |
| IMC | 23,7 % | Hereda los faltantes de estatura **o** peso |
| Age | 3,5 % | Edad desconocida, casi siempre en atletas muy antiguos |
| region | ~0,01 % | Equipos sin país (refugiados, Tuvalu, desconocido) |

**¿Cambia según la época?** Sí, y muchísimo. Antes de 1960 falta la estatura en el **79,1 %** de los registros y el peso en el **82,3 %**; desde 1960 en adelante solo falta en el **5,3 %** y el **5,7 %**. La edad pasa de **13,5 %** de faltantes a **0,5 %**. Esto muestra que los faltantes **no son aleatorios**: dependen del año (los registros históricos no guardaban esas medidas), así que un análisis de estatura, peso o IMC representa sobre todo a los atletas modernos. Llama la atención un pequeño repunte en los 90 (14 %), que probablemente se debe a fuentes incompletas de esa década.

## 9. Preguntas basadas en los datos

In [23]:
# Punto 9: preguntas basadas en los datos
print("(a) Edad promedio (por registro):", round(df["Age"].mean(), 2))
print("    Edad promedio (atleta único, primera aparición):",
      round(df.sort_values("Year").drop_duplicates("ID")["Age"].mean(), 2))
print("(b) Registros con medalla / sin medalla:")
print(df["medalla"].value_counts().rename({1: "Con medalla", 0: "Sin medalla"}))
print("(c) Estatura promedio de medallistas de oro:", round(df.loc[df["Medal"] == "Gold", "Height"].mean(), 2), "cm")
print("(d) Ediciones de los Juegos:", df["Games"].nunique(), "| NOC distintos:", df["NOC"].nunique())
print("(e) Edad mínima:", df["Age"].min(), "| Edad máxima:", df["Age"].max())
print("(f) Registros por temporada:")
print(df["Season"].value_counts())
ediciones_por_atleta = df.groupby("ID")["Games"].nunique()
print("(g) Atletas en una sola edición:", (ediciones_por_atleta == 1).sum(),
      "| en más de una:", (ediciones_por_atleta > 1).sum())

(a) Edad promedio (por registro): 25.56
    Edad promedio (atleta único, primera aparición): 24.45
(b) Registros con medalla / sin medalla:
medalla
Sin medalla    231333
Con medalla     39783
Name: count, dtype: int64
(c) Estatura promedio de medallistas de oro: 177.9 cm
(d) Ediciones de los Juegos: 51 | NOC distintos: 230
(e) Edad mínima: 10.0 | Edad máxima: 97.0
(f) Registros por temporada:
Season
Summer    222552
Winter     48564
Name: count, dtype: int64


(g) Atletas en una sola edición: 98450 | en más de una: 37121


In [24]:
# Atletas con la edad mínima y máxima
df[df["Age"].isin([df["Age"].min(), df["Age"].max()])][["Name", "Sex", "Age", "Games", "Sport", "Event", "Medal"]]

,Name,Sex,Age,Games,Sport,Event,Medal
142882,Dimitrios Loundras,M,10.0,1896 Summer,Gymnastics,"Gymnastics Men's Parallel Bars, Teams",Bronze
257054,John Quincy Adams Ward,M,97.0,1928 Summer,Art Competitions,"Art Competitions Mixed Sculpturing, Statues",Sin medalla


**Respuestas**

- **(a)** La edad promedio es de **25,56 años** por registro. Si contamos cada atleta una sola vez (en su primera participación), baja a **24,45 años**, porque los atletas que repiten aparecen más veces y con más edad.
- **(b)** **39.783** registros terminaron con medalla y **231.333** sin medalla (14,7 % vs 85,3 %).
- **(c)** La estatura promedio de los medallistas de oro es de **177,9 cm**, unos 2,5 cm más que el promedio general (175,3 cm).
- **(d)** Hay **51 ediciones** de los Juegos (29 de verano y 22 de invierno) y **230 NOC** distintos.
- **(e)** La edad más joven es **10 años** (Dimitrios Loundras, gimnasia por equipos en 1896, ganó bronce) y la más vieja es **97 años** (John Quincy Adams Ward, competencias de arte en 1928; las artes fueron parte de los Juegos hasta 1948).
- **(f)** Verano tiene **222.552** registros e invierno **48.564**.
- **(g)** **98.450** atletas (72,6 %) participaron en una sola edición y **37.121** (27,4 %) en más de una.

## 10. Hipótesis y verificación con los datos

Primero escribimos lo que creíamos y luego lo verificamos con `groupby`. Para la parte proporcional usamos la **tasa de medalla** = registros con medalla / registros totales del grupo.

In [25]:
# Función auxiliar: conteo de registros, medallas y tasa de medalla por grupo
def tasa_medalla(data, columna):
    tabla = data.groupby(columna)["medalla"].agg(registros="count", medallas="sum")
    tabla["tasa_%"] = (100 * tabla["medallas"] / tabla["registros"]).round(2)
    return tabla

### (a) Mujeres vs hombres
**Lo que creíamos:** los hombres ganaron más medallas porque han participado mucho más.

In [26]:
# Punto 10a: mujeres vs hombres
t10a = tasa_medalla(df, "Sex")
t10a.to_csv(f"{TABLAS}/p10a_medallas_por_sexo.csv")
t10a

,registros,medallas,tasa_%
Sex,,,
F,74522,11253,15.10
M,196594,28530,14.51


**Verificación:** en número absoluto, los hombres tienen muchas más medallas (**28.530** vs **11.253**). Pero proporcionalmente las mujeres tienen una tasa un poco mayor (**15,10 %** vs **14,51 %**). Es decir, la diferencia absoluta se explica por la diferencia en participación, no porque los hombres ganen más seguido.

### (b) Menores de 18 vs adultos
**Lo que creíamos:** los adultos ganan más medallas, tanto en total como en proporción, porque tienen más experiencia. Excluimos los 9.474 registros sin edad.

In [27]:
# Punto 10b: menores de 18 vs adultos (se excluyen registros sin edad)
con_edad = df[df["Age"].notna()].copy()
con_edad["menor_18"] = np.where(con_edad["Age"] < 18, "Menor de 18", "18 o más")
t10b = tasa_medalla(con_edad, "menor_18")
t10b.to_csv(f"{TABLAS}/p10b_medallas_menores_vs_adultos.csv")
t10b

,registros,medallas,tasa_%
menor_18,,,
18 o más,249134,37872,15.20
Menor de 18,12508,1179,9.43


**Verificación:** se confirma en ambos sentidos. Los adultos tienen **37.872** medallas frente a **1.179** de los menores, y su tasa también es más alta (**15,20 %** vs **9,43 %**). Los menores de 18 compiten sobre todo en deportes como gimnasia y natación, donde hay muchos participantes por prueba.

### (c) Menores de 18 o 35+ vs el resto (18 a 34)
**Lo que creíamos:** los atletas de 18 a 34 años ganan más medallas porque están en su mejor etapa física.

In [28]:
# Punto 10c: (menores de 18 o 35 y más) vs el resto (18 a 34)
con_edad["extremos"] = np.where((con_edad["Age"] < 18) | (con_edad["Age"] >= 35),
                                "Menor de 18 o 35+", "18 a 34")
t10c = tasa_medalla(con_edad, "extremos")
t10c.to_csv(f"{TABLAS}/p10c_medallas_extremos_edad.csv")
t10c

,registros,medallas,tasa_%
extremos,,,
18 a 34,229617,35066,15.27
Menor de 18 o 35+,32025,3985,12.44


**Verificación:** correcto. El grupo de 18 a 34 años tiene **35.066** medallas contra **3.985** de los extremos, y además una tasa mayor (**15,27 %** vs **12,44 %**). La diferencia en proporción es menor que en conteo, porque los mayores de 35 tienen una tasa relativamente alta (ver punto 12).

### (d) Estados Unidos vs el resto de países
**Lo que creíamos:** en total los demás países juntos ganan más, pero USA debe tener una tasa mucho más alta.

In [29]:
# Punto 10d: Estados Unidos vs otros países
df["es_usa"] = np.where(df["NOC"] == "USA", "USA", "Otros países")
t10d = tasa_medalla(df, "es_usa")
t10d.to_csv(f"{TABLAS}/p10d_medallas_usa_vs_otros.csv")
print(t10d)
print()
print("Top 5 NOC por medallas (filas):")
print(df.groupby("NOC")["medalla"].sum().sort_values(ascending=False).head())

              registros  medallas  tasa_%
es_usa                                   
Otros países     252263     34146   13.54
USA               18853      5637   29.90

Top 5 NOC por medallas (filas):
NOC
USA    5637
URS    2503
GER    2165
GBR    2068
FRA    1777
Name: medalla, dtype: int64


**Verificación:** los otros países juntos suman más medallas (**34.146** vs **5.637**), lo cual es lógico porque son más de 200 NOC. Pero USA es, por mucho, el NOC con más medallas (más del doble que la URSS, que es el segundo con 2.503), y su tasa es de **29,90 %**, más del doble que la del resto (**13,54 %**). Casi 3 de cada 10 registros de atletas estadounidenses terminan en medalla.

### (e) Verano vs Invierno
**Lo que creíamos:** en verano se ganan más medallas porque hay más deportes y pruebas.

In [30]:
# Punto 10e: Verano vs Invierno
t10e = tasa_medalla(df, "Season")
t10e["ediciones"] = df.groupby("Season")["Games"].nunique()
t10e["medallas_por_edicion"] = (t10e["medallas"] / t10e["ediciones"]).round(1)
t10e.to_csv(f"{TABLAS}/p10e_medallas_por_temporada.csv")
t10e

,registros,medallas,tasa_%,ediciones,medallas_por_edicion
Season,,,,,
Summer,222552,34088,15.32,29,1175.4
Winter,48564,5695,11.73,22,258.9


**Verificación:** sí. Verano tiene **34.088** registros con medalla frente a **5.695** en invierno, y también una tasa mayor (**15,32 %** vs **11,73 %**). Incluso por edición, en verano se reparten unos 1.175 registros con medalla y en invierno unos 259. Parte de la diferencia en tasa se explica porque en verano hay más deportes de equipo (cada integrante cuenta como medalla).

## 11. Grupos de participación según `n_pruebas`

`n_pruebas` cuenta en cuántas pruebas compitió cada atleta dentro de una misma edición (agrupando por `ID` y `Games`). Para la distribución usamos una fila por atleta-edición, para no contar dos veces al mismo atleta en la misma edición.

In [31]:
# Punto 11: n_pruebas por atleta y edición, y grupo de participación
df["n_pruebas"] = df.groupby(["ID", "Games"])["Event"].transform("count")
df["grupo_participacion"] = pd.cut(df["n_pruebas"], bins=[0, 1, 3, np.inf],
                                   labels=["Grupo 1 (1 prueba)", "Grupo 2 (2-3)", "Grupo 3 (4+)"])

atleta_edicion = df.drop_duplicates(["ID", "Games"])
print("Atletas-edición por grupo:")
print(atleta_edicion["grupo_participacion"].value_counts().sort_index())
print()
print((100 * atleta_edicion["grupo_participacion"].value_counts(normalize=True).sort_index()).round(2))

Atletas-edición por grupo:
grupo_participacion
Grupo 1 (1 prueba)    139986
Grupo 2 (2-3)          39774
Grupo 3 (4+)            7692
Name: count, dtype: int64



grupo_participacion
Grupo 1 (1 prueba)    74.68
Grupo 2 (2-3)         21.22
Grupo 3 (4+)           4.10
Name: proportion, dtype: float64


In [32]:
# Distribución de deportes en cada grupo (% dentro del grupo, top 8 de cada uno)
deporte_grupo = atleta_edicion.groupby("grupo_participacion", observed=True)["Sport"].value_counts(normalize=True).mul(100).round(2)
deporte_grupo.rename("porcentaje").reset_index().to_csv(f"{TABLAS}/p11_deportes_por_grupo_participacion.csv", index=False)
for grupo in atleta_edicion["grupo_participacion"].cat.categories:
    print(grupo)
    print(deporte_grupo[grupo].head(8).to_string())
    print()

Grupo 1 (1 prueba)
Sport
Athletics     17.07
Rowing         7.05
Football       4.81
Wrestling      4.68
Sailing        4.50
Boxing         4.31
Ice Hockey     3.93
Hockey         3.87

Grupo 2 (2-3)
Sport
Athletics               16.02
Swimming                14.41
Fencing                  8.12
Equestrianism            6.65
Cycling                  6.64
Alpine Skiing            6.46
Shooting                 5.60
Cross Country Skiing     4.83

Grupo 3 (4+)
Sport
Gymnastics              48.01
Swimming                12.91
Cross Country Skiing    10.86
Biathlon                 5.08
Alpine Skiing            4.00
Speed Skating            3.85
Shooting                 3.68
Fencing                  2.91



In [33]:
# Número de deportes distintos presentes en cada grupo
atleta_edicion.groupby("grupo_participacion", observed=True)["Sport"].nunique()

grupo_participacion
Grupo 1 (1 prueba)    66
Grupo 2 (2-3)         51
Grupo 3 (4+)          28
Name: Sport, dtype: int64

**Análisis**

- **Grupo 1 (una prueba):** es el más grande, con **139.986** atletas-edición (74,7 %), y aparecen los **66 deportes**. Predominan atletismo (17,1 %), remo, fútbol, lucha, vela y boxeo, que son deportes donde normalmente se compite en una sola prueba o con un equipo fijo.
- **Grupo 2 (2 a 3 pruebas):** **39.774** (21,2 %). Sube mucho la natación (14,4 %), la esgrima, la equitación, el ciclismo y el esquí alpino, deportes con varias distancias o modalidades.
- **Grupo 3 (4 o más pruebas):** es el grupo pequeño, con **7.692** (4,1 %) y solo **28 deportes**. Casi la mitad es **gimnasia (48,0 %)**, porque un gimnasta compite en cada aparato y en el all-around, seguida de natación (12,9 %), esquí de fondo (10,9 %) y biatlón.

Ningún grupo quedó vacío, pero el Grupo 3 es pequeño y está muy concentrado en pocos deportes. Por eso sus resultados dependen casi totalmente de la gimnasia.

## 12. Proporciones por sexo, grupo etario, medidas y medalla

Definimos `grupo_etario` con tres categorías: **Menores** (< 18), **Jóvenes adultos** (18 a 34) y **Mayores** (35 o más).

**¿Qué hicimos con los atletas sin edad?** No los eliminamos: los dejamos en una cuarta categoría, **"Sin edad"** (9.474 registros), para no perder información y poder ver si se comportan distinto. En los puntos 10b y 10c sí los excluimos, porque allí la edad es necesaria para la comparación.

In [34]:
# Punto 12: grupo etario (los registros sin edad quedan como 'Sin edad')
df["grupo_etario"] = pd.cut(df["Age"], bins=[0, 17, 34, np.inf],
                            labels=["Menores", "Jóvenes adultos", "Mayores"])
df["grupo_etario"] = df["grupo_etario"].cat.add_categories("Sin edad").fillna("Sin edad")
df["tiene_medidas"] = np.where(df["Height"].notna() & df["Weight"].notna(),
                               "Con estatura y peso", "Sin estatura o peso")
df["grupo_etario"].value_counts()

grupo_etario
Jóvenes adultos    229617
Mayores             19517
Menores             12508
Sin edad             9474
Name: count, dtype: int64

La primera tabla muestra, **dentro de cada combinación de sexo y grupo etario** (cada fila suma 100 %), qué porcentaje tenía o no estatura y peso, y si ganó o no medalla. La segunda muestra directamente la tasa de medalla.

In [35]:
# Proporción (%) dentro de cada sexo y grupo etario según medidas y medalla
p12 = pd.crosstab([df["Sex"], df["grupo_etario"]],
                  [df["tiene_medidas"], df["medalla"].map({1: "Medalla", 0: "Sin medalla"})],
                  normalize="index").mul(100).round(2)
p12.to_csv(f"{TABLAS}/p12_proporciones_sexo_edad_medidas_medalla.csv")
p12

tiene_medidas       Con estatura y peso             Sin estatura o peso            
medalla                         Medalla Sin medalla             Medalla Sin medalla
Sex grupo_etario                                                                   
F   Menores                        8.01       78.20                1.49       12.30
    Jóvenes adultos               14.97       75.93                1.11        7.99
    Mayores                       11.80       71.98                2.05       14.17
    Sin edad                       0.00        4.25                4.01       91.75
M   Menores                        4.82       65.83                4.38       24.97
    Jóvenes adultos               11.13       65.73                3.85       19.30
    Mayores                        6.10       43.78                8.37       41.75
    Sin edad                       0.17        7.24                7.73       84.86

In [36]:
# Tasa de medalla (%) según si tiene medidas, por sexo y grupo etario
p12_tasa = (df.groupby(["Sex", "grupo_etario", "tiene_medidas"], observed=True)["medalla"]
            .agg(registros="count", tasa_medalla="mean"))
p12_tasa["tasa_medalla"] = (100 * p12_tasa["tasa_medalla"]).round(2)
p12_tasa.to_csv(f"{TABLAS}/p12_tasa_medalla_por_medidas.csv")
p12_tasa.unstack("tiene_medidas")

registros                            tasa_medalla                    
tiene_medidas       Con estatura y peso Sin estatura o peso Con estatura y peso Sin estatura o peso
Sex grupo_etario                                                                                   
F   Menores                        8049                1287                9.29               10.80
    Jóvenes adultos               56255                5634               16.47               12.21
    Mayores                        2407                 466               14.08               12.66
    Sin edad                         18                 406                0.00                4.19
M   Menores                        2241                 931                6.83               14.93
    Jóvenes adultos              128911               38817               14.48               16.62
    Mayores                        8302                8342               12.23               16.70
    Sin edad                        670                8380                2.24                8.35

**Conclusión del punto 12**

- **Registro de medidas:** las mujeres tienen las medidas registradas con mucha más frecuencia (por ejemplo, el 90,9 % de las jóvenes adultas frente al 76,9 % de los hombres jóvenes adultos). Esto pasa porque las mujeres participan sobre todo en Juegos recientes, cuando ya se medía a todos. En los **hombres mayores** solo la mitad (49,9 %) tiene medidas, porque muchos compitieron a comienzos del siglo XX (por ejemplo, en tiro y en competencias de arte). Los registros **sin edad** casi nunca tienen estatura ni peso (más del 90 %): son los registros históricos más incompletos.
- **Medallas:** en ambos sexos, los **jóvenes adultos** tienen la tasa más alta cuando tienen medidas (16,5 % mujeres y 14,5 % hombres), y los **menores** la más baja. Los registros sin edad casi no tienen medallas (tasas de 8,4 % o menos).
- **Detalle interesante:** en los hombres, los que **no tienen medidas** tienen una tasa de medalla **mayor** (por ejemplo, 16,7 % vs 12,2 % en mayores). Eso no significa que no medirse ayude a ganar. Lo que pasa es que esos registros son de los primeros Juegos, donde competían menos países y menos atletas por prueba, así que era más fácil ganar medalla. Es un buen ejemplo de una variable de confusión: la **época**.